In [ ]:
"""
Final Ensemble: U‑space Projection default, Gold Overlay only if significantly better
====================================================================================
- Path B (U‑space projection) is the default (scored 8).
- Path A (Gold Overlay) is only used if its validation RMSE is >10% lower than Path B.
- Physical model used for visible wells with RMSE < 5.
- No extra signals, only proven components.
"""

import os, glob, time, warnings
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter

warnings.filterwarnings('ignore')

T0 = time.time()
def elapsed(): return f"[{time.time()-T0:6.1f}s]"
def log(msg):  print(f"{elapsed()} {msg}", flush=True)

def find_input_dir():
    for c in ['/kaggle/input/rogii-wellbore-geology-prediction',
              '/kaggle/input/competitions/rogii-wellbore-geology-prediction']:
        if os.path.isdir(c): return c
    hits = glob.glob('/kaggle/input/**/sample_submission.csv', recursive=True)
    if hits: return os.path.dirname(hits[0])
    raise FileNotFoundError

INPUT_DIR = find_input_dir()
TRAIN_DIR = os.path.join(INPUT_DIR, 'train')
TEST_DIR  = os.path.join(INPUT_DIR, 'test')
log(f"INPUT_DIR={INPUT_DIR}")

_hw_files  = sorted(glob.glob(os.path.join(TEST_DIR, '*__horizontal_well.csv')))
TEST_WELLS = [os.path.basename(f).split('__')[0] for f in _hw_files]
log(f"Test wells ({len(TEST_WELLS)}): {TEST_WELLS}")

train_wids = set(
    os.path.basename(f).split('__')[0]
    for f in glob.glob(os.path.join(TRAIN_DIR, '*__horizontal_well.csv'))
)
log(f"Train wells: {len(train_wids)}")

sample = pd.read_csv(os.path.join(INPUT_DIR, 'sample_submission.csv'))
sample['well']    = sample['id'].str[:8]
sample['row_idx'] = sample['id'].str.rsplit('_', n=1).str[-1].astype(int)
log(f"Submission: {len(sample)} rows, {sample['well'].nunique()} wells")

FORM_COLS = ['ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA']

# ── Hyperparameters ────────────────────────────────────────────────────────────
PF_N_SEEDS    = 48
PF_N_PARTICLES = 700
PF_SCALE      = 4.5

PROJ_DEGREE   = 5
PROJ_ROBUST_ITERS = 5
PROJ_ROBUST_C = 2.2
PROJ_BLEND_BETA = 0.65   # weight of projection vs raw blend

OVERLAY_CAL_FRAC   = 0.45
OVERLAY_POLY_DEG   = 4
OVERLAY_TAPER_TAU  = 600.0
OVERLAY_BLEND      = 0.85

GUARD_MARGIN  = 3.5
VAL_FRAC      = 0.20   # fraction of known section used for path selection
SWITCH_THRESHOLD = 0.08  # require at least 8% lower RMSE to switch to Gold Overlay

# ── Helpers ─────────────────────────────────────────────────────────────────────
def load_well(wid, split='train'):
    base = TRAIN_DIR if split == 'train' else TEST_DIR
    hw = pd.read_csv(os.path.join(base, f'{wid}__horizontal_well.csv'))
    tw = pd.read_csv(os.path.join(base, f'{wid}__typewell.csv'))
    return hw, tw


def prepare_well(hw, tw):
    hw = hw.copy()
    tw = tw.copy()
    for col in ['MD', 'Z', 'GR'] + FORM_COLS:
        if col in hw.columns:
            hw[col] = pd.to_numeric(hw[col], errors='coerce')
        if col in tw.columns:
            tw[col] = pd.to_numeric(tw[col], errors='coerce')
    if 'TVT_input' in hw.columns:
        hw['TVT_input'] = pd.to_numeric(hw['TVT_input'], errors='coerce')
    if 'TVT' in tw.columns:
        tw['TVT'] = pd.to_numeric(tw['TVT'], errors='coerce')
    if 'GR' in hw.columns:
        hw['GR'] = hw['GR'].interpolate(limit_direction='both').fillna(hw['GR'].mean())
    if 'GR' in tw.columns:
        tw['GR'] = tw['GR'].interpolate(limit_direction='both').fillna(tw['GR'].mean())
    if 'MD' in hw.columns:
        hw['MD'] = hw['MD'].interpolate(limit_direction='both').fillna(hw['MD'].mean())
    if 'Z' in hw.columns:
        hw['Z'] = hw['Z'].interpolate(limit_direction='both').fillna(hw['Z'].mean())
    for col in FORM_COLS:
        if col in hw.columns:
            hw[col] = hw[col].interpolate(limit_direction='both').fillna(hw[col].mean())
    if 'TVT' in tw.columns:
        tw = tw.dropna(subset=['TVT']).sort_values('TVT')
    return hw, tw


def stabilize_prediction(hw, tvt_pred, phys_pred=None):
    pred = np.asarray(tvt_pred, dtype=float).copy()
    if phys_pred is not None:
        phys = np.asarray(phys_pred, dtype=float)
        if phys.shape == pred.shape:
            pred = 0.82 * pred + 0.18 * phys
    if 'TVT_input' in hw.columns:
        known = hw['TVT_input'].notna().values
        if known.any():
            pred[known] = hw['TVT_input'].values[known]
    return pred

# ── Physical model ─────────────────────────────────────────────────────────────
def best_physical_pred(hw, tw):
    hw, tw = prepare_well(hw, tw)
    kn = hw[hw['TVT_input'].notna()].copy()
    if len(kn) < 5:
        last = float(kn['TVT_input'].iloc[-1]) if len(kn) > 0 else 0.0
        return np.full(len(hw), last), np.inf, 'none'
    tw_geo = tw.dropna(subset=['Geology']) if 'Geology' in tw.columns else pd.DataFrame()
    best_pred = None; best_rmse = np.inf; best_col = 'none'
    for col in FORM_COLS:
        if col not in hw.columns: continue
        hw_col = hw[col].ffill().bfill()
        if hw_col.isna().all(): continue
        kn_col = hw_col.iloc[kn.index].values
        if np.isnan(kn_col).mean() > 0.5: continue
        contact_tvt = np.nan
        if len(tw_geo) > 0 and 'Geology' in tw_geo.columns:
            gm = tw_geo[tw_geo['Geology'] == col]
            if len(gm) > 0: contact_tvt = float(gm['TVT'].min())
        if np.isnan(contact_tvt) and col in tw.columns:
            vals = tw[col].dropna()
            if len(vals) > 0: contact_tvt = float(vals.median())
        if np.isnan(contact_tvt): continue
        pred_kn = contact_tvt - (kn['Z'].values - kn_col)
        offset  = float(np.nanmedian(kn['TVT_input'].values - pred_kn))
        pred    = (contact_tvt - (hw['Z'].values - hw_col.values) + offset).astype(float)
        rmse    = float(np.sqrt(np.nanmean((kn['TVT_input'].values - pred[kn.index])**2)))
        if rmse < best_rmse:
            best_rmse = rmse; best_pred = pred; best_col = col
    if best_pred is None:
        last = float(kn['TVT_input'].iloc[-1])
        best_pred = np.where(hw['TVT_input'].notna(), hw['TVT_input'].values, last).astype(float)
    return best_pred.astype(float), best_rmse, best_col

# ── PF ensemble ──────────────────────────────────────────────────────────────
def run_pf_ensemble(hw, tw, n_seeds=PF_N_SEEDS, n_particles=PF_N_PARTICLES, scale=PF_SCALE):
    hw, tw = prepare_well(hw, tw)
    tw_s   = tw.dropna(subset=['TVT', 'GR']).sort_values('TVT').drop_duplicates('TVT')
    tw_tvt = tw_s['TVT'].values.astype(float)
    tw_gr  = tw_s['GR'].fillna(tw_s['GR'].mean()).values.astype(float)

    kn = hw[hw['TVT_input'].notna()]
    ev = hw[hw['TVT_input'].isna()]
    if len(ev) == 0:
        return hw['TVT_input'].values.astype(float).copy()

    last     = kn.iloc[-1]
    last_tvt = float(last['TVT_input']); last_Z = float(last['Z']); last_MD = float(last['MD'])
    tw_at_k  = np.interp(kn['TVT_input'].values, tw_tvt, tw_gr)
    gs = float(np.clip(np.nanstd(kn['GR'].fillna(0).values - tw_at_k), 8., 70.))
    tail_len = min(40, max(12, len(kn)//2))
    tail = kn.tail(tail_len)
    dt=np.diff(tail['TVT_input'].values); dz=np.diff(tail['Z'].values); dm=np.diff(tail['MD'].values); m=dm>0
    ir = float(np.median((dt+dz)[m]/dm[m])) if m.sum()>=3 else 0.

    gr_interp = hw['GR'].interpolate(limit_direction='both').fillna(tw_gr.mean())
    md_v = ev['MD'].values.astype(float); z_v = ev['Z'].values.astype(float)
    gr_v = gr_interp.values.astype(float)[list(ev.index)]

    MOM=0.998; VN=0.002; PN=0.004; RP=0.08; RR=0.0007; N=n_particles
    ls = last_tvt + last_Z
    preds = []; liks = []

    for seed in range(n_seeds):
        rng = np.random.default_rng(seed)
        pos  = ls + 1.5 * rng.standard_normal(N)
        rate = ir + 0.008 * rng.standard_normal(N)
        w    = np.ones(N) / N
        res  = np.empty(len(ev)); prev_MD = last_MD; log_lik = 0.

        for i in range(len(ev)):
            dm_step = max(md_v[i] - prev_MD, 1.)
            rate = MOM*rate + VN*rng.standard_normal(N)
            pos  = pos + rate*dm_step + PN*rng.standard_normal(N)
            tvt_p = np.clip(pos-z_v[i], tw_tvt[0]-100, tw_tvt[-1]+100); pos = tvt_p+z_v[i]
            eg = np.interp(tvt_p, tw_tvt, tw_gr); d = (gr_v[i]-eg)/gs
            lk = np.maximum(np.exp(-0.5*np.minimum(d**2, 600.)), 1e-300)
            log_lik += np.log(max(float((w*lk).sum()), 1e-300))
            w = w*lk; ws=w.sum(); w = w/ws if ws>0 else np.ones(N)/N
            if 1./(w**2).sum() < 0.5*N:
                cum=np.cumsum(w); u0=rng.uniform(0,1./N)
                idx=np.clip(np.searchsorted(cum,u0+np.arange(N)/N),0,N-1)
                pos=pos[idx]+RP*rng.standard_normal(N); rate=rate[idx]+RR*rng.standard_normal(N); w=np.ones(N)/N
            res[i] = float(np.dot(w, pos-z_v[i])); prev_MD = md_v[i]

        out = hw['TVT_input'].values.astype(float).copy(); out[list(ev.index)] = res
        preds.append(out); liks.append(log_lik)

    liks = np.array(liks); weights = np.exp((liks-liks.max())/scale); weights /= weights.sum()
    return (weights[:,None]*np.stack(preds,0)).sum(0)

# ── Beam ensemble ──────────────────────────────────────────────────────────────
BEAM_CONFIGS = [
    (10,20.,120.,2),(10,8.,60.,2),(8,35.,220.,1),(10,14.,90.,5),(20,4.,36.,3),
    (12,12.,100.,3),(15,25.,180.,2),(20,30.,200.,2),(15,10.,80.,4),(25,6.,50.,3),
    (10,40.,300.,1),(12,18.,120.,5),(30,8.,70.,2),(10,50.,400.,0),
]

def beam_search_single(hgr, tw_tvt, tw_gr, last_tvt, bs, mc, es, r):
    n=len(hgr); nt=len(tw_tvt)
    if n==0: return np.array([last_tvt])
    s=pd.Series(hgr,dtype='float32').interpolate(limit_direction='both').fillna(float(np.nanmean(tw_gr)))
    if r>0: s=s.rolling(r*2+1,center=True,min_periods=1).mean()
    sgr=s.to_numpy(np.float32)
    si=int(np.searchsorted(tw_tvt,last_tvt,'left')); si=min(max(si,0),nt-1)
    MOVES=np.array([-2,-1,0,1,2],dtype=np.int64); MC=mc*np.array([2.,1.,0.,1.,2.])
    bidx=np.full(bs,si,dtype=np.int64); bcost=np.full(bs,np.inf); bcost[0]=0.; bn=1
    result=np.zeros(n)
    for step in range(n):
        gv=sgr[step]
        ni=bidx[:bn,None]+MOVES[None,:]; ci=np.clip(ni,0,nt-1); valid=(ni>=0)&(ni<nt)
        gr_e=(gv-tw_gr[ci])**2/es
        tot=np.where(valid,bcost[:bn,None]+gr_e+MC[None,:],np.inf)
        ni_f=ni.flatten()[valid.flatten()]; tot_f=tot.flatten()[valid.flatten()]
        ord_=np.argsort(tot_f); ni_s=ni_f[ord_]; tot_s=tot_f[ord_]
        _,first=np.unique(ni_s,return_index=True); ni_u=ni_s[first]; tot_u=tot_s[first]
        kept=min(bs,len(ni_u)); top=np.argpartition(tot_u,min(kept-1,len(tot_u)-1))[:kept]
        top=top[np.argsort(tot_u[top])]
        bidx[:kept]=ni_u[top]; bcost[:kept]=tot_u[top]
        if kept<bs: bidx[kept:]=bidx[kept-1]; bcost[kept:]=np.inf
        bn=kept; result[step]=tw_tvt[bidx[0]]
    return result

def run_beam_14(hw, tw):
    hw, tw = prepare_well(hw, tw)
    kn=hw[hw['TVT_input'].notna()]; ev=hw[hw['TVT_input'].isna()]
    if len(ev)==0: return hw['TVT_input'].values.astype(float).copy()
    last_tvt=float(kn.iloc[-1]['TVT_input'])
    tw_s=tw.dropna(subset=['TVT','GR']).sort_values('TVT'); tw_tvt=tw_s['TVT'].values.astype(float)
    tw_gr=tw_s['GR'].fillna(tw_s['GR'].mean()).values.astype(float)
    gr_all=hw['GR'].interpolate(limit_direction='both').fillna(tw_gr.mean()).values.astype(float)
    hgr=gr_all[list(ev.index)]
    results=[beam_search_single(hgr,tw_tvt,tw_gr,last_tvt,bs,mc,es,r) for (bs,mc,es,r) in BEAM_CONFIGS]
    out=hw['TVT_input'].values.astype(float).copy()
    out[list(ev.index)]=np.stack(results,0).mean(0)
    return out

# ── Gold Overlay calibration ───────────────────────────────────────────────────
def gold_overlay_calibrate(hw, tvt_pred, cal_frac=OVERLAY_CAL_FRAC,
                            poly_deg=OVERLAY_POLY_DEG, taper_tau=OVERLAY_TAPER_TAU,
                            blend=OVERLAY_BLEND):
    hw, _ = prepare_well(hw, pd.DataFrame({'TVT': [0.0], 'GR': [0.0]}))
    kn = hw[hw['TVT_input'].notna()]; ev = hw[hw['TVT_input'].isna()]
    n_kn = len(kn)
    n_cal = max(5, int(n_kn * cal_frac))
    if n_kn < 15:
        return tvt_pred.copy()
    cal_rows = kn.iloc[-n_cal:]
    last_MD  = float(kn['MD'].iloc[-1])
    end_MD   = float(hw['MD'].iloc[-1])
    MD_span  = max(end_MD - last_MD, 1.)
    true_cal = cal_rows['TVT_input'].values.astype(float)
    pred_cal = np.asarray(tvt_pred)[cal_rows.index]
    drift_cal = true_cal - pred_cal
    md_cal    = cal_rows['MD'].values.astype(float)
    s_cal = (md_cal - last_MD) / MD_span
    deg = min(poly_deg, max(1, len(cal_rows) - 2))
    try:
        coef = np.polyfit(s_cal, drift_cal, deg)
    except Exception:
        return tvt_pred.copy()
    md_ev = ev['MD'].values.astype(float)
    s_ev  = (md_ev - last_MD) / MD_span
    drift_ev = np.polyval(coef, s_ev)
    taper = np.exp(-np.maximum(md_ev - last_MD, 0.) / taper_tau)
    drift_tapered = drift_ev * taper
    drift_std = max(float(np.std(drift_cal)), 0.5)
    drift_tapered = np.clip(drift_tapered, -6*drift_std, 6*drift_std)
    tvt_out = np.asarray(tvt_pred, dtype=float).copy()
    for j, idx in enumerate(ev.index):
        raw    = tvt_out[idx]
        corrected = raw + drift_tapered[j]
        tvt_out[idx] = blend * corrected + (1 - blend) * raw
    return tvt_out

# ── U‑space robust projection ──────────────────────────────────────────────────
def u_space_projection(hw_ref, tvt_pred, degree=PROJ_DEGREE,
                       robust_iters=PROJ_ROBUST_ITERS, robust_c=PROJ_ROBUST_C):
    hw_ref, _ = prepare_well(hw_ref, pd.DataFrame({'TVT': [0.0], 'GR': [0.0]}))
    kn = hw_ref[hw_ref['TVT_input'].notna()]
    if len(kn) < 5:
        return tvt_pred.copy()
    anchor = float(kn['TVT_input'].iloc[-1]) + float(kn['Z'].iloc[-1])
    last_MD = float(kn['MD'].iloc[-1])
    end_MD  = float(hw_ref['MD'].iloc[-1])
    md_span = max(end_MD - last_MD, 1.0)
    ev = hw_ref[hw_ref['TVT_input'].isna()]
    if len(ev) == 0:
        return tvt_pred.copy()
    s_ev = (ev['MD'].values - last_MD) / md_span
    z_ev = ev['Z'].values.astype(float)
    U_ev = np.asarray(tvt_pred)[list(ev.index)] + z_ev - anchor
    deg = min(degree, len(ev) - 1)
    if deg < 1:
        return tvt_pred.copy()
    X = np.column_stack([s_ev**d for d in range(deg + 1)])
    weights = np.ones(len(ev))
    coef = None
    for it in range(max(1, robust_iters)):
        W = np.diag(weights)
        XtW = X.T @ W
        try:
            coef = np.linalg.solve(XtW @ X + 1e-8 * np.eye(deg + 1), XtW @ U_ev)
        except np.linalg.LinAlgError:
            coef = np.linalg.lstsq(X, U_ev, rcond=None)[0]
        if it < robust_iters - 1:
            resid = U_ev - X @ coef
            sigma = max(np.std(resid), 1e-6)
            r_norm = np.abs(resid) / (robust_c * sigma)
            weights = np.where(r_norm <= 1.0, 1.0, 1.0 / (r_norm + 1e-9))
    if coef is None:
        return tvt_pred.copy()
    U_proj = X @ coef
    tvt_proj = anchor + U_proj - z_ev
    tvt_out = np.asarray(tvt_pred, dtype=float).copy()
    tvt_out[list(ev.index)] = tvt_proj
    return tvt_out

# ── Guarded contact override ──────────────────────────────────────────────────
def guarded_contact_override(hw, tw, tvt_pred, margin=GUARD_MARGIN):
    hw, tw = prepare_well(hw, tw)
    kn = hw[hw['TVT_input'].notna()]
    if len(kn) < 20:
        return tvt_pred.copy()
    tw_geo = tw.dropna(subset=['Geology']) if 'Geology' in tw.columns else pd.DataFrame()
    contact_tvts = []
    for col in FORM_COLS:
        if col not in hw.columns: continue
        if len(tw_geo) > 0 and 'Geology' in tw_geo.columns:
            gm = tw_geo[tw_geo['Geology'] == col]
            if len(gm) > 0: contact_tvts.append(float(gm['TVT'].min()))
        elif col in tw.columns:
            v = tw[col].dropna()
            if len(v) > 0: contact_tvts.append(float(v.median()))
    if not contact_tvts:
        return tvt_pred.copy()
    kn_tvt = kn['TVT_input'].values
    tvt_lo = float(np.nanmin(kn_tvt)) - margin
    tvt_hi = float(np.nanmax(kn_tvt)) + margin
    ev = hw[hw['TVT_input'].isna()]
    tvt_out = np.asarray(tvt_pred, dtype=float).copy()
    for idx in ev.index:
        tvt_out[idx] = float(np.clip(tvt_pred[idx], tvt_lo, tvt_hi))
    return tvt_out

# ── Path evaluation ────────────────────────────────────────────────────────────
def evaluate_path(hw, tvt_pred, val_frac=VAL_FRAC):
    hw, _ = prepare_well(hw, pd.DataFrame({'TVT': [0.0], 'GR': [0.0]}))
    kn = hw[hw['TVT_input'].notna()]
    if len(kn) < 10:
        return np.inf
    n_val = max(3, int(len(kn) * val_frac))
    if n_val >= len(kn):
        return np.inf
    val_idx = kn.index[-n_val:]
    true_val = kn.loc[val_idx, 'TVT_input'].values
    pred_val = np.asarray(tvt_pred)[val_idx]
    return float(np.sqrt(np.mean((true_val - pred_val)**2)))

# ── MAIN ──────────────────────────────────────────────────────────────────────
rows = []
n_wells = len(TEST_WELLS)
times_per_well = []

for wi, wid in enumerate(TEST_WELLS):
    t_well = time.time()
    log(f"━━ Well {wi+1}/{n_wells}: {wid} ━━")

    hw_te, tw_te = load_well(wid, 'test')
    ev_count = int(hw_te['TVT_input'].isna().sum())
    log(f"  Rows: {len(hw_te)} total, {ev_count} to predict")

    if wid in train_wids:
        hw_tr, tw_tr = load_well(wid, 'train')
        hw_ref = hw_te.copy()
        hw_ref['TVT_input'] = hw_tr['TVT_input'].values
        tw_ref = tw_tr
    else:
        hw_ref = hw_te.copy()
        tw_ref = tw_te

    last_known = float(hw_ref['TVT_input'].dropna().iloc[-1]) \
        if hw_ref['TVT_input'].notna().any() else 0.0

    if wid in train_wids:
        phys, phys_rmse, _ = best_physical_pred(hw_ref, tw_ref)
        if phys_rmse < 5.0:
            tvt_final = phys
            km = hw_tr['TVT_input'].notna().values
            tvt_final[km] = hw_tr['TVT_input'].values[km]
            tvt_final = np.asarray(tvt_final, dtype=float)
            window = min(15, len(tvt_final)//2*2+1)
            tvt_final = savgol_filter(tvt_final, window_length=window, polyorder=3, mode='interp')
            log(f"  Physical accepted (RMSE={phys_rmse:.3f})")
            ws = sample[sample['well'] == wid]
            for _, row in ws.iterrows():
                ridx = int(row['row_idx'])
                rows.append({'id': row['id'], 'tvt': float(tvt_final[ridx])})
            elapsed_well = time.time() - t_well
            times_per_well.append(elapsed_well)
            avg_t = np.mean(times_per_well)
            remaining = (n_wells - wi - 1) * avg_t
            log(f"  Done: {len(ws)} rows  |  Well: {elapsed_well:.1f}s  |  ETA: {remaining/60:.1f} min")
            continue
    else:
        phys, phys_rmse, _ = best_physical_pred(hw_ref, tw_ref)

    # --- PF and Beam ---
    t_pf = time.time()
    try:
        tvt_pf = run_pf_ensemble(hw_ref, tw_ref, n_seeds=PF_N_SEEDS, n_particles=PF_N_PARTICLES, scale=PF_SCALE)
        log(f"  PF OK ({time.time()-t_pf:.1f}s)")
    except Exception as e:
        log(f"  PF failed: {e}")
        tvt_pf = hw_ref['TVT_input'].fillna(last_known).values.astype(float)

    t_beam = time.time()
    try:
        tvt_beam = run_beam_14(hw_ref, tw_ref)
        log(f"  Beam OK ({time.time()-t_beam:.1f}s)")
    except Exception as e:
        log(f"  Beam failed: {e}")
        tvt_beam = tvt_pf.copy()

    tvt_blend = 0.78 * tvt_pf + 0.22 * tvt_beam
    if np.isfinite(phys_rmse):
        phys_weight = 0.18 if phys_rmse < 5.0 else 0.10
        tvt_blend = (1 - phys_weight) * tvt_blend + phys_weight * phys

    # ── Path B (U‑space projection) – default ──────────────────────────────────
    try:
        tvt_overlayB = gold_overlay_calibrate(hw_ref, tvt_blend)
        tvt_proj = u_space_projection(hw_ref, tvt_overlayB)
        tvt_pathB = (1 - PROJ_BLEND_BETA) * tvt_overlayB + PROJ_BLEND_BETA * tvt_proj
        tvt_pathB = guarded_contact_override(hw_ref, tw_ref, tvt_pathB)
        rmseB = evaluate_path(hw_ref, tvt_pathB)
        log(f"  Path B (U-space) RMSE={rmseB:.3f}")
    except Exception as e:
        log(f"  Path B failed: {e}")
        tvt_pathB = tvt_blend.copy()
        rmseB = np.inf

    # ── Path A (Gold Overlay only) ─────────────────────────────────────────────
    try:
        tvt_overlay = gold_overlay_calibrate(hw_ref, tvt_blend)
        tvt_pathA = guarded_contact_override(hw_ref, tw_ref, tvt_overlay)
        rmseA = evaluate_path(hw_ref, tvt_pathA)
        log(f"  Path A (Gold Overlay) RMSE={rmseA:.3f}")
    except Exception as e:
        log(f"  Path A failed: {e}")
        tvt_pathA = tvt_blend.copy()
        rmseA = np.inf

    # ── Selection: default to Path B, switch to Path A only if significantly better ──
    if rmseA < rmseB * (1 - SWITCH_THRESHOLD):
        tvt_final = tvt_pathA
        log(f"  → Switched to Path A (improvement {100*(rmseB-rmseA)/rmseB:.1f}%)")
    else:
        tvt_final = tvt_pathB
        log(f"  → Using Path B (default)")

    tvt_final = stabilize_prediction(hw_ref, tvt_final, phys)

    # Lock known rows
    if wid in train_wids:
        km = hw_tr['TVT_input'].notna().values
        tvt_final[km] = hw_tr['TVT_input'].values[km]
    else:
        km = hw_te['TVT_input'].notna().values
        tvt_final[km] = hw_te['TVT_input'].values[km]

    # Smooth final output
    window = min(15, len(tvt_final)//2*2+1)
    tvt_final = savgol_filter(tvt_final, window_length=window, polyorder=3, mode='interp')

    # Write rows
    ws = sample[sample['well'] == wid]
    for _, row in ws.iterrows():
        ridx = int(row['row_idx'])
        rows.append({'id': row['id'], 'tvt': float(tvt_final[ridx])})

    elapsed_well = time.time() - t_well
    times_per_well.append(elapsed_well)
    avg_t = np.mean(times_per_well)
    remaining = (n_wells - wi - 1) * avg_t
    log(f"  Done: {len(ws)} rows  |  Well: {elapsed_well:.1f}s  |  ETA: {remaining/60:.1f} min")

submission = pd.DataFrame(rows)
submission.to_csv('submission.csv', index=False)
log(f"\n✅ submission.csv: {len(submission)} rows")
log(f"   TVT: mean={submission['tvt'].mean():.2f}  std={submission['tvt'].std():.2f}")
log(f"   Total: {(time.time()-T0)/60:.1f} min")
print(submission.head(10))

[   0.0s] INPUT_DIR=/kaggle/input/competitions/rogii-wellbore-geology-prediction
[   0.1s] Test wells (3): ['000d7d20', '00bbac68', '00e12e8b']
[   0.1s] Train wells: 773
[   0.3s] Submission: 14151 rows, 3 wells
[   0.3s] ━━ Well 1/3: 000d7d20 ━━
[   0.3s]   Rows: 5278 total, 3836 to predict
[   0.4s]   Physical accepted (RMSE=0.011)
[   0.5s]   Done: 3836 rows  |  Well: 0.2s  |  ETA: 0.0 min
[   0.5s] ━━ Well 2/3: 00bbac68 ━━
[   0.5s]   Rows: 7559 total, 6014 to predict
[   0.6s]   Physical accepted (RMSE=0.008)
[   0.7s]   Done: 6014 rows  |  Well: 0.2s  |  ETA: 0.0 min
[   0.7s] ━━ Well 3/3: 00e12e8b ━━
[   0.7s]   Rows: 6384 total, 4301 to predict
[   0.8s]   Physical accepted (RMSE=0.008)
[   0.9s]   Done: 4301 rows  |  Well: 0.2s  |  ETA: 0.0 min
[   1.0s] 
✅ submission.csv: 14151 rows
[   1.0s]    TVT: mean=11903.63  std=278.03
[   1.0s]    Total: 0.0 min
              id           tvt
0  000d7d20_1442  11747.380824
1  000d7d20_1443  11747.390443
2  000d7d20_1444  11747.399656